# GWTC-5 dark-siren population inference with `tempest`

To install `tempest` run this command `pip install git+https://github.com/mtagliazucchi/tempest.git`

In [ ]:
import numpy as np
import scipy
import jax.numpy as jnp
import h5py

from CHIMERA import data
from CHIMERA.population import cosmo, mass, rate, population
from CHIMERA import selection_function, hyperlikelihood

import tempest as tp

## 1. Load GWTC-5 posterior samples and injections

Data files (`GWTC5_242CBC_FAR0.25_PE2048.h5`, its PE-prior `.npy`, and
`GWTC5_injections_far0.25.h5`) live in `data/` alongside this notebook.


In [ ]:
DATA_DIR = "data/"

pe_gw = data.load_gw_pe_samples(
    DATA_DIR + "GWTC5_242CBC_FAR0.25_PE2048.h5",
    parameters=["m1det", "m2det", "dL"],
    return_struct=True,
)
pe_prior = jnp.load(DATA_DIR + "GWTC5_242CBC_FAR0.25_PE2048prior.npy")
pe_gw = pe_gw.update(pe_prior=pe_prior)

f_inj = h5py.File(DATA_DIR + "GWTC5_injections_far0.25.h5")
Ngen = float(f_inj.attrs["Ngen"])  # total number of generated injections
Tobs = float(f_inj.attrs["Tobs"])  # observing time [yr]

theta_inj_det = data.load_injection_data(
    DATA_DIR + "GWTC5_injections_far0.25.h5",
    snr_cut=None,
    key_mapping={"snr": "snr", "log_pdraw": "log_pdraw"},
    frame="detector",
)
sel_fcn = selection_function(theta_inj_det, N_inj=Ngen)

print(f"Loaded GWTC-5 events and injections.")
print(f"Tobs = {Tobs:.3f} yr, Ngen = {Ngen:.3e}")

## 2. Population model


In [ ]:
cosmo_model = cosmo.flrw(Om0=0.3065, z_max=5.)
mass_model = mass.paired.bpl_dip_three_peaks() 
rate_model = rate.madau_dickinson()

pop_model = population(cosmo_model, mass_model, rate_model, scale_free=True, Tobs=Tobs, R0=1)

hyperlike = hyperlikelihood(
    theta_gw_det=pe_gw,
    population=pop_model,
    z_grids_res=64,            # coarser grid than the production run, for speed
    selection_function=sel_fcn,
    pe_neff=2.0,
    inj_neff=None,             # defaults to 5 * Nevents
    kind_kde="fft",
    kernel="gaussian",
    kde_bw=None,               # Scott's rule
)

## 3. Priors and likelihood wrapper for `tempest`

`tempest.Sampler` expects a `prior_transform(u)` mapping the unit cube to
physical parameters, and a `log_likelihood(params)` function. 


In [ ]:
def uniform(low, high):
 return scipy.stats.uniform(loc=low, scale=high-low)

def loguniform(low, high):
  return scipy.stats.loguniform(a=low, b=high)

def norm(loc, scale):
  return scipy.stats.norm(loc=loc, scale=scale)

def truncnorm(loc, scale, low, high):
  return scipy.stats.truncnorm(low, high, loc=loc, scale=scale)

def beta(alpha, beta, low=0., high=1.):
    return scipy.stats.beta(alpha, beta, loc=low, scale=high-low)

# Priors
priors = {"H0": uniform(10.,200.),
  "alpha_1": uniform(-4,12.),
  "alpha_2": uniform(-4,12.),
  "beta_bottom": uniform(-4.,12.),
  "beta_top": uniform(-4.,12.),
  "mu_g1": uniform(5.,150.),       
  "sigma_g1": uniform(0.4,10.),               
  "mu_g2":uniform(5.0, 150.),
  "sigma_g2":uniform(0.4, 15.),
  "mu_g3":uniform(5.0, 150.),
  "sigma_g3":uniform(0.4, 15.),
  "lambda_g":uniform(0,1),
  "lambda_1":beta(alpha=1, beta=2, low=0, high=1),
  "lambda_2":uniform(0,1),
  "bottomsmooth": loguniform(0.01,1.),
  "topsmooth": loguniform(0.001,1.),
  "leftdip": uniform (1.5,3.0),
  "rightdip": uniform(5.0, 9.0),
  "leftdipsmooth": loguniform(0.01,2.0),
  "rightdipsmooth": loguniform(0.01,2.0),
  "deep": uniform(0., 1.),  
  "m_low": uniform(0.4,1.4),
  "m_high": uniform(50.,300.),
  "gamma": uniform(0.,12.),
  "kappa": uniform(0.,6.),
  "zp": uniform(0.,4.)
}
params_keys = list(priors.keys())
n_dim = len(params_keys)
prior_dists = [priors[k] for k in params_keys]

def prior_transform(u):
    return np.array([dist.ppf(u[i]) for i, dist in enumerate(prior_dists)])

def log_likelihood(params):
    hyperparams = {k: params[i] for i, k in enumerate(params_keys)}
    log_like_evs, N_exp, neff_inj, log_like = hyperlike.compute_all(**hyperparams)
    return log_like, log_like_evs, N_exp, neff_inj  # (logL, blobs...)

## 4. Run `tempest`

Persistent Sampling (Karamanis & Seljak 2025) anneals from the prior
(`beta=0`) to the full posterior (`beta=1`) while tracking the evidence.


In [ ]:
import glob, os

dir_chain = './states/'
chain_label = "gwtc5_bpl2p"
restart = True

if restart:
  state_files = glob.glob(f"{dir_chain}/{chain_label}_*.state")
  latest_state = None
  final_state = f"{dir_chain}/{chain_label}_final.state"
  if os.path.exists(final_state):
    latest_state = final_state
  elif state_files:
    numeric_state_files = [f for f in state_files if not f.endswith('_final.state')]
    if numeric_state_files:
      latest_state = max(numeric_state_files,
                        key=lambda x: int(x.split('_')[-1].split('.')[0]))
else:
  latest_state=None

if latest_state is not None:
  print("restarting from", latest_state)



sampler = tp.Sampler(
    prior_transform=prior_transform,
    log_likelihood=log_likelihood,
    n_dim=n_dim,
    n_particles=256,
    n_steps=5,
    resample="syst",
    random_state=0,
    output_dir=dir_chain,
    output_label=chain_label
)

#sampler.run(n_total=4096, save_every=1, resume_state_path=latest_state) # --> uncomment this to run

## 5. Posterior and evidence


In [ ]:
import xarray as xr
import arviz as az
from cornetto import corner
import matplotlib.pyplot as plt

samples, weights, logl, blobs = sampler.posterior(resample=True, return_blobs=True)
samples_dict = {k: samples[:, i] for i, k in enumerate(priors)}

log_like_evs_blob = blobs[:,0]
N_exp_blob = blobs[:,1]
neff_inj_blob = blobs[:,2]

logZ, logZerr = sampler.evidence()
logZerr = str('none') if logZerr is None else logZerr

print(f"log-evidence: {logZ:.2f}")

n_chains = 1
n_draws = len(samples)
n_events = np.stack(log_like_evs_blob).shape[-1]
posterior_ds = xr.Dataset(
  {k: (['chain', 'draw'], v.reshape(n_chains, n_draws)) for k, v in samples_dict.items()}
)
sample_stats_ds = xr.Dataset({
  'log_likelihood': (['chain', 'draw'], logl.reshape(n_chains, n_draws)),
  'weights': (['chain', 'draw'], weights.reshape(n_chains, n_draws)),
  'log_like_evs': (['chain', 'draw', 'event'], np.stack(log_like_evs_blob).reshape(n_chains, n_draws, n_events)),
  'N_exp': (['chain', 'draw'], np.asarray(N_exp_blob).reshape(n_chains, n_draws)),
  'Neff_inj': (['chain', 'draw'], np.asarray(neff_inj_blob).reshape(n_chains, n_draws))
})

idata = az.InferenceData(
  posterior=posterior_ds,
  sample_stats=sample_stats_ds
)
idata.attrs['log_evidence'] = float(logZ)
idata.attrs['log_evidence_err'] = logZerr
file = f"{dir_chain}/{chain_label}_final.state"
idata.to_netcdf(f"{file.replace('.state', '.nc')}")



fig, axes = corner(
    samples_dict,
    show_titles=True
)
plt.show()